# Demo 01: fit an encoding model and select the layer (Monkey R)

The encoding model of a site under a DNN is a ridge readout of PCA-reduced layer features:

    image -> backbone -> layer activations (flattened) -> PCA (fit on the training images) -> RidgeCV -> response

The pipeline is `scripts/encoding/neural_regression_massprod_VVS_multimonkey_20250428-20250430_PCA4All.py`; this
notebook runs its building blocks (`core.layer_hook_utils.featureFetcher`, `neural_regress.regress_lib`) on
ResNet-50 for a few candidate layers and applies the layer-selection rule of
`scripts/encoding/neural_regress_yaml_export.py`: for each site, the layer with the highest held-out test $R^2$
among the `pca750` + `RidgeCV` fits.

**Scale.** `FULL_SCALE = False` uses 4 candidate layers, 300 of the 774 training images and 200 PCs;
the paper used all 16 bottleneck blocks, all 774 training images and 750 PCs (`pca750`). The 195 held-out test
images are always all used, so the test $R^2$ is comparable with the cached full-scale value.

In [1]:
import sys
sys.path[:0] = ['..', 'notebooks']        # notebooks/nbsetup.py holds the setup shared by every notebook
from nbsetup import setup

OUT_DIR = setup(needs='source')          # single-thread BLAS, Agg backend, repo on the path, data check
import os, sys, time
from pathlib import Path
from pnc import paths
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from pnc.utils import apply_figure_style
apply_figure_style(); plt.rcParams['figure.dpi'] = 90
torch.set_num_threads(max(1, os.cpu_count() // 2)); torch.manual_seed(0); np.random.seed(0)
print('source_data       =', paths.source_data())
print('preprocessed_data =', paths.preprocessed_data())
from IPython.display import display

source_data       = /private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/source_data
preprocessed_data = /private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/preproc_build


In [2]:
FULL_SCALE = False
MONKEY, UNIT = 'red', 9                                   # aIT site used throughout the demos
from core.model_load_utils import MODEL_LAYER_FILTERS
if FULL_SCALE:
    N_TRAIN, PCA_N, LAYERS = None, 750, None                # all 774 training images, pca750, all 16 Bottleneck blocks
else:
    N_TRAIN, PCA_N = 300, 200
    LAYERS = ['.layer2.Bottleneck3', '.layer3.Bottleneck5', '.layer4.Bottleneck0', '.layer4.Bottleneck2']
BATCH = 16
RIDGE_ALPHAS = [1E-4, 1E-3, 1E-2, 1E-1, 1, 10, 100, 1E3, 1E4, 1E5, 1E6, 1E7, 1E8, 1E9]   # the production grid
print(f'FULL_SCALE={FULL_SCALE}: n_train={N_TRAIN or 774}, PCA={PCA_N}, layers={LAYERS or "all Bottleneck blocks"}')

FULL_SCALE=False: n_train=300, PCA=200, layers=['.layer2.Bottleneck3', '.layer3.Bottleneck5', '.layer4.Bottleneck0', '.layer4.Bottleneck2']


## 1. Responses and the train / test split

`repavg/response_peak` (969 x 64 channels, standardized) and the fixed split of `encoding_stimuli_split_seed0.csv`.
The readouts are fitted to *all* channels at once (`RidgeCV(alpha_per_target=True)`: one ridge penalty per
channel); the target sites are simply columns of the fit.

In [3]:
from pnc.utils import load_encoding_hdf5, MONKEY_UNITS
import _demo_utils as U

stim_names, resp_mat, _, _ = load_encoding_hdf5(MONKEY)
split = pd.read_csv(paths.source_data() / 'stimuli_encoding' / 'encoding_stimuli_split_seed0.csv')
assert (split.stimulus_name.values == stim_names).all()
is_train = split.is_train.values.astype(bool)
train_all, test_all = np.where(is_train)[0], np.where(~is_train)[0]
rng = np.random.RandomState(0)
train_sel = train_all if N_TRAIN is None else np.sort(rng.choice(train_all, N_TRAIN, replace=False))   # reduced scale subsamples training images only
keep = np.concatenate([train_sel, test_all])                # images that enter this notebook
train_idx = np.arange(len(train_sel))                        # positions of the training images inside `keep`
resp = resp_mat[keep]
image_paths = [str(U.stimulus_path(n)) for n in stim_names[keep]]
print(f'{len(train_sel)} training + {len(test_all)} held-out test images; responses {resp.shape}')
print('target sites of Monkey R:', MONKEY_UNITS[MONKEY])

300 training + 195 held-out test images; responses (495, 64)
target sites of Monkey R: [0, 2, 9, 15, 19]


## 2. Layer features with `featureFetcher`

`featureFetcher` registers forward hooks by module name (`.layer4.Bottleneck0` = first bottleneck block of stage 4);
`record_features` pushes the images through the frozen backbone with the model's own preprocessing and returns
`{layer: (n_images, C, H, W)}`. The production run records every bottleneck block (`MODEL_LAYER_FILTERS['resnet50']`).

`neural_regress.regress_lib` (which holds `record_features`, `transform_features2Xdict_new` and `sweep_regressors`)
imports `imageio`, an optional dependency of the GPU scripts that the reference environment does not install, so
the three functions are restated here in compact form; the logic (batching, PCA fit on the training rows, RidgeCV
with one alpha per channel, train / test $R^2$) is the same.

In [4]:
from core.layer_hook_utils import featureFetcher
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class ImagePathDataset(Dataset):                                  # core.dataset_utils.ImagePathDataset, minimal
    def __init__(self, paths, transform):
        self.paths, self.transform = paths, transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert('RGB')), 0.0

def record_features(model, fetcher, dataset, batch_size=20, device='cpu'):   # regress_lib.record_features
    feat_col = {}
    for imgs, _ in DataLoader(dataset, batch_size=batch_size, shuffle=False):
        with torch.no_grad():
            model(imgs.to(device))
        for key in fetcher.activations:
            feat_col.setdefault(key, []).append(fetcher[key].cpu())
    return {k: torch.cat(v, dim=0) for k, v in feat_col.items()}

model, transform = U.load_backbone('resnet50', device='cpu')
fetcher = featureFetcher(model, input_size=(3, 224, 224), device='cpu', print_module=False, store_device='cpu')
if LAYERS is None:                                               # full scale: every bottleneck block, as in the production sweep
    LAYERS = [n for n in fetcher.module_names.values() if MODEL_LAYER_FILTERS['resnet50'](n)]
for name in LAYERS:
    fetcher.record(name, ingraph=False, store_device='cpu')       # detached activations: fitting needs no gradient
dataset = ImagePathDataset(image_paths, transform)
t0 = time.time()
feat_dict = record_features(model, fetcher, dataset, batch_size=BATCH, device='cpu')
fetcher.cleanup()
print(f'feature extraction: {time.time() - t0:.0f} s')
for k, v in feat_dict.items():
    print(f'  {k:22s} {tuple(v.shape)}  -> {int(np.prod(v.shape[1:]))} features per image')

/private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/env/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/env/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


FeatureFetcher hooks all freed
feature extraction: 63 s
  .layer2.Bottleneck3    (495, 512, 28, 28)  -> 401408 features per image
  .layer3.Bottleneck5    (495, 1024, 14, 14)  -> 200704 features per image
  .layer4.Bottleneck0    (495, 2048, 7, 7)  -> 100352 features per image
  .layer4.Bottleneck2    (495, 2048, 7, 7)  -> 100352 features per image


## 3. PCA + RidgeCV, one fit per layer

`transform_features2Xdict_new` flattens each layer, fits the PCA on the training images only and projects every
image (`Xdict[(layer, 'pcaN')]`); `sweep_regressors` fits the regressors on the training rows and reports the
train / test $R^2$ averaged over channels (`RidgeCV.score`). The keys follow the library convention
`((layer, dimred), regressor)` so that `neural_regress.regress_eval_lib` can evaluate the fits. (The GPU dual PCA
solver of the production script is replaced by sklearn's PCA here -- same decomposition, CPU.)

In [5]:
from copy import deepcopy
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from neural_regress.regress_eval_lib import compute_pred_dict_D2_per_unit

def transform_features2Xdict(feat_dict, layers, n_components, train_split_idx):   # regress_lib.transform_features2Xdict_new, pca branch
    Xdict, tfm_dict = {}, {}
    for layer in layers:
        featmat = feat_dict[layer].flatten(start_dim=1).numpy()
        pca = PCA(n_components=n_components).fit(featmat[train_split_idx])   # basis from the training rows only; test images stay held out
        Xdict[(layer, f'pca{n_components}')] = pca.transform(featmat)
        tfm_dict[(layer, f'pca{n_components}')] = pca
    return Xdict, tfm_dict

def sweep_regressors(Xdict, y_all, regressors, regressor_names, train_split_idx):   # regress_lib.sweep_regressors, RidgeCV branch
    idx_train = train_split_idx; idx_test = np.setdiff1d(np.arange(len(y_all)), idx_train)
    summary, models = {}, {}
    for xtype, X_all in Xdict.items():
        for estim, label in zip(regressors, regressor_names):
            clf = deepcopy(estim.fit(X_all[idx_train], y_all[idx_train]))   # copy: the same estimator object is refitted for the next layer
            summary[(xtype, label)] = dict(alpha=np.median(clf.alpha_), train_score=clf.score(X_all[idx_train], y_all[idx_train]),
                                           test_score=clf.score(X_all[idx_test], y_all[idx_test]), n_feat=X_all.shape[1])
            models[(xtype, label)] = clf
    return pd.DataFrame(summary).T, models

t0 = time.time()
Xdict, Xtfmer = transform_features2Xdict(feat_dict, LAYERS, PCA_N, train_idx)
print(f'PCA: {time.time() - t0:.0f} s')
regressors = [RidgeCV(alphas=RIDGE_ALPHAS, alpha_per_target=True)]
result_df, fit_models = sweep_regressors(Xdict, resp, regressors, ['RidgeCV'], train_idx)
scores = compute_pred_dict_D2_per_unit(fit_models, Xdict, resp, idx_train=train_idx)   # per-channel train / test R^2 and predictions
print('(alpha = median over the 64 channels of the per-channel ridge penalty)')
print(result_df.round(3))

PCA: 28 s
using provided idx_train as train set, and the rest as test set (N_train=300, N_test=195)
(alpha = median over the 64 channels of the per-channel ridge penalty)
                                           alpha  train_score  test_score  \
(.layer2.Bottleneck3, pca200) RidgeCV   100000.0        0.301       0.044   
(.layer3.Bottleneck5, pca200) RidgeCV    10000.0        0.300       0.075   
(.layer4.Bottleneck0, pca200) RidgeCV   100000.0        0.330       0.095   
(.layer4.Bottleneck2, pca200) RidgeCV  1000000.0        0.295       0.101   

                                       n_feat  
(.layer2.Bottleneck3, pca200) RidgeCV   200.0  
(.layer3.Bottleneck5, pca200) RidgeCV   200.0  
(.layer4.Bottleneck0, pca200) RidgeCV   200.0  
(.layer4.Bottleneck2, pca200) RidgeCV   200.0  


## 4. Held-out $R^2$ per layer per site, and the layer-selection rule

`D2_per_unit_test_dict[((layer, dimred), regressor)]` holds the held-out $R^2$ of every channel. The production rule
(`neural_regress_yaml_export.py`) restricts to `dimred == 'pca750'` and `RidgeCV` and takes the layer with the
best test score, separately for every target site.

In [6]:
D2_test = scores['D2_per_unit_test_dict']
keys = [k for k in D2_test if k[0][1] == f'pca{PCA_N}' and k[1] == 'RidgeCV']   # the selection rule considers only this dimred + regressor
r2 = pd.DataFrame({k[0][0]: D2_test[k] for k in keys})              # channels x layers
r2_sites = r2.loc[MONKEY_UNITS[MONKEY]]                              # rows are channel indices, so the site ids index directly
r2_sites.index.name = 'site'
selected = r2_sites.idxmax(axis=1)
print('held-out R^2 per candidate layer (rows = target sites):')
print(r2_sites.round(3).to_string())
print('\nselected layer (argmax held-out R^2):')
print(selected.to_string())

held-out R^2 per candidate layer (rows = target sites):
      .layer2.Bottleneck3  .layer3.Bottleneck5  .layer4.Bottleneck0  .layer4.Bottleneck2
site                                                                                    
0                   0.145                0.142                0.278                0.306
2                   0.090                0.202                0.212                0.232
9                   0.221                0.387                0.420                0.494
15                  0.235                0.392                0.419                0.446
19                  0.138                0.278                0.298                0.341

selected layer (argmax held-out R^2):
site
0     .layer4.Bottleneck2
2     .layer4.Bottleneck2
9     .layer4.Bottleneck2
15    .layer4.Bottleneck2
19    .layer4.Bottleneck2


## 5. Compare with the cached layer selection (`preprocessed_data/layer_selection.pkl`)

The cache was built by `scripts/preprocessing/build_layer_selection.py` from the full-scale sweeps. It is a dict
keyed by `(monkey, model, unit)`; each entry stores the ordered candidate `layers`, their relative depth, the
per-layer held-out $R^2$ `curve`, and the selected `peak_layer` / `peak_r2` (plus recording area and depth for the
anatomy figure). The extra key `_probe_profiles` holds the Neuropixels depth / area profiles.

In [7]:
from pnc.preproc import loader as L
LS = L.load_layer_selection()
print('keys:', len(LS), '   examples:', [k for k in LS][:3], '...')
entry = LS[(MONKEY, 'resnet50', UNIT)]
for k, v in entry.items():
    v_desc = f'{type(v).__name__}[{len(v)}]' if isinstance(v, (list, np.ndarray)) else repr(v)
    print(f'  {k:14s} {v_desc}')
print(f"\ncached peak layer for (red, resnet50, {UNIT}): {entry['peak_layer']}  (test R^2 = {entry['peak_r2']:.3f})")
print(f"this notebook, {'full' if FULL_SCALE else 'reduced'} scale:      {selected[UNIT]}  (test R^2 = {r2_sites.loc[UNIT].max():.3f})")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(entry['layers'], entry['curve'], '-o', ms=3, label='cached full-scale sweep (pca750, 774 train images)')
ours = [r2_sites.loc[UNIT][l] if l in r2_sites.columns else np.nan for l in entry['layers']]   # layers this run did not fit are left blank
ax.plot(entry['layers'], ours, 's', ms=6, color='C3', label=f'this notebook (pca{PCA_N}, {len(train_sel)} train images)')
ax.axvline(entry['layers'].index(entry['peak_layer']), color='k', lw=0.6, ls='--')
ax.set(ylabel='held-out $R^2$', title=f'Monkey R site {UNIT}, ResNet-50 layer sweep'); ax.legend(fontsize=7)
plt.setp(ax.get_xticklabels(), rotation=60, ha='right', fontsize=6); plt.tight_layout(); display(plt.gcf()); plt.close()

keys: 251    examples: ['_probe_profiles', ('red', 'AlexNet_training_seed_01', 0), ('red', 'AlexNet_training_seed_01', 2)] ...
  layers         list[16]
  reldepth       ndarray[16]
  curve          ndarray[16]
  peak_idx       13
  peak_reldepth  0.8666666666666667
  peak_layer     '.layer4.Bottleneck0'
  peak_r2        0.7468237139780225
  n_layers       16
  area           np.str_('aIT')
  depth          nan
  ventral_ord    7

cached peak layer for (red, resnet50, 9): .layer4.Bottleneck0  (test R^2 = 0.747)
this notebook, reduced scale:      .layer4.Bottleneck2  (test R^2 = 0.494)


<Figure size 630x270 with 1 Axes>

## 6. Predicted vs measured on the held-out images

The readout for site 9 at its selected layer, evaluated on the 195 test images that never entered the PCA or the
ridge fit.

In [8]:
layer_sel = selected[UNIT]
key = ((layer_sel, f'pca{PCA_N}'), 'RidgeCV')
pred_all = scores['pred_dict'][key]                      # (n_images, 64) predictions of every channel
test_idx = np.setdiff1d(np.arange(len(keep)), train_idx)
x, y = pred_all[test_idx, UNIT], resp[test_idx, UNIT]
r2_val = 1 - np.sum((y - x) ** 2) / np.sum((y - y.mean()) ** 2)   # R^2 as RidgeCV.score defines it, not the squared correlation
fig, ax = plt.subplots(figsize=(3.6, 3.4))
ax.scatter(x, y, s=10, alpha=0.7)
lim = [min(x.min(), y.min()) - 0.2, max(x.max(), y.max()) + 0.2]; ax.plot(lim, lim, 'k--', lw=0.6)
ax.set(xlabel=f'predicted ({layer_sel}, pca{PCA_N}, RidgeCV)', ylabel='measured response (z)',
       title=f'site {UNIT}, held-out: r = {np.corrcoef(x, y)[0, 1]:.2f}, $R^2$ = {r2_val:.2f}')
plt.tight_layout(); display(plt.gcf()); plt.close()

<Figure size 324x306 with 1 Axes>

## 7. Exporting the readout for synthesis

Synthesis, attacks and gradients need the fitted model as a differentiable torch graph.
`neural_regress.sklearn_torchify_lib` converts the sklearn objects: `PCA_torch` (the basis, exported as a JIT script
`..._Xtfmer_..._JITscript.pt`) and `LinearLayer_from_sklearn` (an `nn.Linear` with the ridge weights, exported as
`..._readout_....pth`). The post-hoc pickles in `image_pca_projections/` carry the single-site slice
(`readout_vec`, `readout_bias`) that the later demos use.

In [9]:
from neural_regress.sklearn_torchify_lib import LinearLayer_from_sklearn, PCA_torch
readout = LinearLayer_from_sklearn(fit_models[key])      # nn.Linear(PCA_N -> 64 channels)
pca_t = PCA_torch(Xtfmer[key[0]])                        # nn.Module: flatten -> center -> project
feats = feat_dict[layer_sel][test_idx[:8]]               # raw layer activations: the torch chain must reproduce the sklearn predictions
with torch.no_grad():
    torch_pred = readout(pca_t(feats))[:, UNIT].numpy()
print('torch chain  :', torch_pred.round(3))
print('sklearn chain:', x[:8].round(3))
print('max |difference|:', np.abs(torch_pred - x[:8]).max())
print(readout, '\nreadout vector for site', UNIT, ':', tuple(readout.weight[UNIT].shape), ' bias', float(readout.bias[UNIT]))

torch chain  : [0.101 0.137 0.333 0.275 0.401 0.29  0.616 0.354]
sklearn chain: [0.101 0.137 0.333 0.275 0.401 0.29  0.616 0.354]
max |difference|: 2.0465194549013077e-06
Linear(in_features=200, out_features=64, bias=True) 
readout vector for site 9 : (200,)  bias -0.032943446189165115


/var/folders/b5/z8z92dmn6lnfsg0r78yj48pr0000gn/T/ipykernel_10787/1844664884.py:10: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runnerx/miniforge3/conda-bld/libtorch_1769327475468/work/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  print(readout, '\nreadout vector for site', UNIT, ':', tuple(readout.weight[UNIT].shape), ' bias', float(readout.bias[UNIT]))
